# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

[Dataset schema URL](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

> This workbook walks through loading dataset metadata, examining available record sets/fields and their `@id`s, extracting tabular data, simple EDA, and visualization.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
md = dataset.metadata
print(f"{md.name}: {md.description}")

# Optional: inspect available record sets (if exposed by metadata)
if hasattr(md, 'recordSet') and md.recordSet:
    print('Available record sets:')
    for rs in md.recordSet:
        print(f"- @id: {rs['@id']}, Type: {rs.get('@type', 'Unknown')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema typically defines `recordSet` elements for each tabular dataset (table), with each field (column) referenced by its `@id`. We'll programmatically enumerate available record sets and fields using their `@id`.


In [ ]:
from pprint import pprint

# Let's collect all record sets and their fields' @ids
record_sets = []
fields_by_recordset = {}

# Parse all record sets from dataset (via Croissant API or metadata)
if hasattr(md, 'recordSet') and md.recordSet:
    for rs in md.recordSet:
        rs_id = rs['@id']
        record_sets.append(rs_id)
        # Find fields for this record set, if available
        fields = []
        if 'field' in rs and rs['field']:
            for field in rs['field']:
                fields.append(field['@id'])
        fields_by_recordset[rs_id] = fields
    print("Record Sets and their Field `@id`s:")
    for rs_id in record_sets:
        print(f"- RecordSet @id: {rs_id}")
        print("  Fields:")
        for f_id in fields_by_recordset.get(rs_id, []):
            print(f"    + Field @id: {f_id}")
else:
    print("No record sets found in metadata.")
# Enumerate a few records for each RecordSet by @id
for rs_id in record_sets:
    print(f"\nSample records from RecordSet @id: {rs_id}")
    for rec in dataset.records(record_set=rs_id):
        pprint(rec)
        break  # Show only one example

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract all available record sets, referencing by their Croissant-defined `@id`.

In [ ]:
# Extract data from every record set
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"RecordSet @id: {rs_id} DataFrame shape: {df.shape}")
    print("Columns:", df.columns.tolist())
    print(df.head(2))

# For demonstration, select the first record set for deeper analysis
if record_sets:
    main_rs_id = record_sets[0]
    print(f"\nUsing RecordSet @id: {main_rs_id} for example analysis")
    print(dataframes[main_rs_id].head())
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations: Removing outliers, transforming distributions, grouping by attributes. All entities referenced by their Croissant `@id`.


In [ ]:
# Set up EDA for the main record set
import numpy as np

df = dataframes[main_rs_id] if main_rs_id else pd.DataFrame()

# Identify numeric fields by inspecting columns (all are referenced by `@id`)
numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field_id}")
    # Set threshold (e.g., to filter 'age at diagnosis' or hormone levels; use median as demo)
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the chosen numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical field, if present (all referenced by @id)
    possible_cat_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
    if possible_cat_fields:
        group_field = possible_cat_fields[0]
        print(f"Grouping by field @id: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print("Mean of numeric field per group:")
        print(grouped_df.head())
else:
    print("No numeric fields available for EDA in the selected RecordSet.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll use matplotlib and seaborn to plot a histogram and a boxplot for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for the numeric field if available
if numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of field @id: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot
    plt.figure(figsize=(5,3))
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f"Boxplot of field @id: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If a group field is available, plot group means
    if 'group_field' in locals():
        plt.figure(figsize=(7,4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field_id])
        plt.title(f"Mean of {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No numeric fields to visualize in selected RecordSet.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded dataset metadata and reviewed available record sets using Croissant `@id` references.
- Extracted tabular data for each record set, inspecting their fields and example records.
- Conducted basic EDA on a numeric field, filtering, normalizing, and grouping by categorical attribute (all referenced via their Croissant `@id`).
- Visualized numeric data distributions and group summary statistics.

This exploratory process demonstrates how the Croissant schema and the `mlcroissant` library enable robust, transparent extraction and processing of FAIR dataset packages.

> For any downstream analysis, always refer and document the Croissant `@id`s used for tables and fields to ensure reproducibility and validation against source metadata.